# Deliverable 3 — Real Data, Ingestion & EDA

**Author:** Sahith Yamani  
**Branch:** `team-02/sahith-yamani`  
**Deep-dive:** Leakage — review every suspect field and explain whether it is a leak, an ID, or a usable input.

This notebook covers all seven areas in the Team 02 README. It combines analysis code for local execution with a qualitative draft and places to record my own observations.

**Status:** Authored without execution. No empirical results or model scores are claimed. Sections labelled **Local conclusion** must be completed from my own inspection before submission.

**NDA handling:** The files stay in the repository's ignored `data/raw/` directory. Run this notebook only locally. Do not share output tables, data values, screenshots, or charts. Keep committed Markdown and the hand-off qualitative. Restart and clear every output before saving a version for git. This notebook never saves extracts, cleaned files, or plots.


## Preparation — confirm the prediction task

Before deciding which fields leak, confirm with Akshith/Aidan what the target means, what one record represents, and when a forecast must be made. Ask Kimberly/the lead for the agreed exploration/validation boundary. Also clarify which event each `SF_LOAD_TMS` field records.

The configuration below deliberately starts unconfirmed. Structural and missingness checks can run first. The target-based analysis stops until the target, row meaning, forecast point, and exploration boundary are supplied.

`EXPLORATION_BEFORE` is the first departure month held out from target-based exploration. The proposed chronological boundary must be agreed with the team; it is not a hard-coded split chosen from this dataset. If the team adopts a different evaluation scheme, adapt the selection cell with them before running it.

The proposed target is `OD_PAX`, based on the team discussion. This is not confirmation of whether the field represents booked, flown, estimated, or forecast passengers.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

TARGET_COLUMN = "OD_PAX"
PREDICTION_TASK_CONFIRMED = False
ROW_GRAIN_DESCRIPTION = ""
FORECAST_POINT = ""
SF_LOAD_TMS_EVENT = ""
EXPLORATION_BEFORE = None

# Leave the confirmation flag False until the team confirms the task and split.
# Describe the record meaning and forecast point in words above.
# Set EXPLORATION_BEFORE to the agreed first held-out departure month.


In [ ]:
# Support launching Jupyter from either the repository or its notebook folder.
repo_root = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "AGENTS.md").is_file()
        and (folder / "DATA.md").is_file()
    ),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")

data_dir = repo_root / "data" / "raw"
pax = pd.read_csv(data_dir / "AA_PAX_FORECASTING.csv")
airports = pd.read_csv(data_dir / "AA_AIRPORT_INFO.csv")

required_pax_columns = {
    "DEP_MONTH", "DEP_YEAR", "ORG_AIRPRT", "DES_AIRPRT",
    "OD_PAX", "OD_REV_USD", "SF_LOAD_TMS",
}
required_airport_columns = {"AIRPRT_CD", "CNTRY_CD", "STATE_PROVNC_CD"}
if not required_pax_columns.issubset(pax.columns):
    raise ValueError("Passenger schema differs from the discussed schema; review it locally.")
if not required_airport_columns.issubset(airports.columns):
    raise ValueError("Airport schema differs from the discussed schema; review it locally.")

print("Both files loaded locally. No sample records are displayed.")


## 1. Shape and types

**What I am checking:** the dimensions of each file, the storage types, the meaning of each column, and whether the types match those meanings. Exact dimensions stay in local outputs.

The passenger file contains departure-period fields, origin/destination identifiers, passenger and revenue measures, and a timestamp. A possible row counter needs confirmation as an export index. The airport file contains identification/location fields, coordinate components, physical measurements, and a timestamp.

| Field/group | Meaning to verify | Type consideration |
|---|---|---|
| `DEP_MONTH`, `DEP_YEAR` | Departure period and its relationship to the forecast | Check valid calendar values and integer-like storage before building dates. |
| `ORG_AIRPRT`, `DES_AIRPRT` | Route identifiers and their code system | Treat identifiers as labels; numeric storage does not make them measurements. |
| `OD_PAX` | Exact passenger measure and aggregation | Previously observed as floating-point storage; confirm units and target role. |
| `OD_REV_USD` | Revenue definition, period, and availability | Confirm numeric storage and whether adjustments/negative values have meaning. |
| `SF_LOAD_TMS` | Event represented, timezone, and availability | Date-looking text is not necessarily a parsed datetime. |
| Airport names, codes, hemisphere fields | Labels, code systems, and directions | Categorical/identifier meaning can differ from physical dtype. |
| Coordinate components, runway/elevation fields | Units, definitions, and historical availability | Confirm that numeric fields are measurements; do not guess unfamiliar abbreviations. |

Ask the lead/PM about ambiguous meanings rather than inferring them from an abbreviation.


In [ ]:
for label, frame in [("Passenger", pax), ("Airport", airports)]:
    print(label)
    print("Shape:", frame.shape)
    frame.info(verbose=True, show_counts=True)
    display(pd.DataFrame({"stored_type": frame.dtypes.astype(str)}))

index_candidates = [
    column for column in pax.columns
    if str(column).strip().lower().startswith("unnamed:")
    or str(column).strip().lower() == "index"
]
print("Possible exported-index fields to review locally:", index_candidates)


In [ ]:
# Validate possible date conversions without modifying either loaded frame.
parsed_load_time = pd.to_datetime(pax["SF_LOAD_TMS"], errors="coerce")
load_time_parse_failures = (
    pax["SF_LOAD_TMS"].notna() & parsed_load_time.isna()
)

departure_year = pd.to_numeric(pax["DEP_YEAR"], errors="coerce")
departure_month = pd.to_numeric(pax["DEP_MONTH"], errors="coerce")
valid_period_fields = (
    departure_year.notna()
    & departure_month.notna()
    & departure_year.mod(1).eq(0)
    & departure_month.mod(1).eq(0)
    & departure_month.between(1, 12)
)
departure_period = pd.to_datetime(
    {
        "year": departure_year.where(valid_period_fields),
        "month": departure_month.where(valid_period_fields),
        "day": 1,
    },
    errors="coerce",
)

display(pd.Series({
    "nonmissing_load_values_that_failed_parsing": load_time_parse_failures.sum(),
    "records_without_a_valid_departure_period": departure_period.isna().sum(),
}))
# Parsing does not establish the timestamp's business meaning or timezone.


**Draft takeaway:** Therefore, I should interpret identifiers as labels, validate calendar fields before constructing dates, and confirm what the timestamp records. A floating-point passenger measure does not automatically need conversion to integer storage. Parsing a timestamp does not establish that it was available at forecast time.

**Local conclusion — complete after inspecting:** Describe what one row represents, any inappropriate stored types, any failed date parsing, and the field definitions still awaiting confirmation. Use words only.


## 2. Missingness

**What I am checking:** which columns have gaps, their extent in local summaries, and whether gaps appear associated with geography or another relevant grouping.

In my earlier local check, pandas did not detect missing entries in the passenger file, while the airport state/province field had gaps. The cells below recheck every column rather than assuming those observations cover the whole reference file. Also confirm documented placeholder codes and CSV missing-value conventions; absence of recognized nulls is not proof of complete information.


In [ ]:
for label, frame in [("Passenger", pax), ("Airport", airports)]:
    missing_summary = pd.DataFrame({
        "missing_count": frame.isna().sum(),
        "missing_percent": frame.isna().mean().mul(100),
    }).sort_values("missing_count", ascending=False)
    print(label)
    display(missing_summary)


In [ ]:
state_missing_by_country = (
    airports.assign(state_missing=airports["STATE_PROVNC_CD"].isna())
    .groupby("CNTRY_CD", dropna=False)["state_missing"]
    .agg(
        airport_records="size",
        missing_records="sum",
        missing_share="mean",
    )
    .sort_values("missing_share", ascending=False)
)
display(state_missing_by_country)

if state_missing_by_country["missing_share"].nunique() > 1:
    print("Observed missingness rates vary between country groups.")
else:
    print("Observed missingness rates are the same across country groups.")


**Draft takeaway:** Therefore, I should investigate geographic patterns before selecting a handling rule. A missing state/province code does not by itself establish that the airport has no applicable subdivision. I should distinguish unknown, unavailable, and not-applicable information with the lead. Country-level differences are descriptive evidence; they do not prove a missingness mechanism.

My provisional handling proposal is to preserve airport records while investigating. If a categorical field is retained, an explicitly documented missing category may be appropriate. I will not average identifier codes, label every gap as international, or drop airports solely because this field is missing. Learned imputation must eventually be fitted on training data only.

**Local conclusion — complete after inspecting:** State which fields actually have gaps, whether a clear grouping pattern is visible, and the proposed per-field handling with its justification. Keep rates and counts in local outputs.


## 3. Distributions and outliers

**What I am checking:** target shape, spread, unusual observations, and relevant numerical measurements. I am not assuming the reference write-up's distribution findings are my own.

The next cell deliberately requires the confirmed task and a team-agreed chronological exploration boundary. It excludes held-out departure periods from target-aware exploration. This implements a proposed future-period forecasting setup only after team confirmation; it does not decide the forecast horizon or evaluation scheme on its own.


In [ ]:
if not PREDICTION_TASK_CONFIRMED:
    raise ValueError("Confirm the target, row meaning, forecast point, and split with the team first.")
if not ROW_GRAIN_DESCRIPTION.strip() or not FORECAST_POINT.strip():
    raise ValueError("Record the confirmed row meaning and forecast point in the setup cell.")
if EXPLORATION_BEFORE is None:
    raise ValueError("Set the team-agreed first held-out departure month in EXPLORATION_BEFORE.")
if TARGET_COLUMN not in pax or not pd.api.types.is_numeric_dtype(pax[TARGET_COLUMN]):
    raise ValueError("Confirm the numeric target name and validate its type before plotting.")
if departure_period.isna().any():
    raise ValueError("Resolve invalid or missing departure periods before selecting exploration records.")

exploration_cutoff = pd.Timestamp(EXPLORATION_BEFORE)
if pd.isna(exploration_cutoff):
    raise ValueError("The exploration boundary must be a valid departure month.")
if exploration_cutoff != exploration_cutoff.to_period("M").to_timestamp():
    raise ValueError("Use the first day of the first held-out departure month.")

exploration_mask = departure_period.lt(exploration_cutoff)
if not exploration_mask.any() or exploration_mask.all():
    raise ValueError("The boundary must leave both exploration and held-out records.")

eda_pax = pax.loc[exploration_mask].copy()
eda_period = departure_period.loc[exploration_mask]
print("Exploration records selected. Held-out target values are not used below.")


In [ ]:
# Revenue is inspected as a suspect field, not approved as an input feature.
outcome_measures = list(dict.fromkeys([TARGET_COLUMN, "OD_REV_USD"]))
for column in outcome_measures:
    if not pd.api.types.is_numeric_dtype(eda_pax[column]):
        print("Validate this measure's numeric type locally before plotting:", column)
        continue
    values = eda_pax[column].dropna()
    if values.empty:
        print("No nonmissing exploration values available for:", column)
        continue
    print(column)
    display(values.describe())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    values.plot.hist(bins=30, ax=axes[0], title=f"{column}: distribution")
    values.plot.box(ax=axes[1], title=f"{column}: spread")
    plt.tight_layout()
    plt.show()


In [ ]:
# A boxplot-style screen flags unusual values; it does not identify errors.
outlier_review = []
for column in outcome_measures:
    if not pd.api.types.is_numeric_dtype(eda_pax[column]):
        continue
    values = eda_pax[column].dropna()
    if values.empty:
        continue
    lower_quartile, upper_quartile = values.quantile([0.25, 0.75])
    interquartile_range = upper_quartile - lower_quartile
    if interquartile_range > 0:
        flags = (
            values.lt(lower_quartile - 1.5 * interquartile_range)
            | values.gt(upper_quartile + 1.5 * interquartile_range)
        )
        flagged_count = flags.sum()
        flagged_share = flags.mean()
    else:
        flagged_count = pd.NA
        flagged_share = pd.NA
    outlier_review.append({
        "measure": column,
        "iqr_screen_available": interquartile_range > 0,
        "flagged_count": flagged_count,
        "flagged_share": flagged_share,
        "negative_count": values.lt(0).sum(),
        "zero_count": values.eq(0).sum(),
    })
display(pd.DataFrame(outlier_review))


In [ ]:
# Candidate physical measurements are described separately; no join is assumed.
for column in ["LNGST_RUNWAY_FT_QTY", "ELEVATN_FT_QTY"]:
    if column not in airports or not pd.api.types.is_numeric_dtype(airports[column]):
        continue
    values = airports[column].dropna()
    if values.empty:
        continue
    print(column)
    display(values.describe())
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    values.plot.hist(bins=30, ax=axes[0], title=f"{column}: distribution")
    values.plot.box(ax=axes[1], title=f"{column}: spread")
    plt.tight_layout()
    plt.show()


**Draft takeaway:** Therefore, I should separate unusual observations from confirmed data errors. Large observations may be legitimate; negative or zero values require interpretation using the measure's definition. I will not remove, cap, or transform observations solely because a plot looks skewed. Any transformation or outlier policy should be justified with Aidan/the lead and evaluated appropriately.

**Local conclusion — complete after inspecting:** Describe the observed shape, possible subgroups or tails, whether unusual observations have been validated, and the proposed handling. Explain the consequence for modeling. No numerical results or plot screenshots belong in this Markdown.


## 4. Leakage risk, specific to this dataset

**Question for every field:** Would this exact value have existed when the forecast needed to be made?

- **Target versus inputs:** If confirmed as the target, `OD_PAX` remains the outcome to predict and must be excluded from its own predictors. Excluding it from inputs does not mean deleting the target.
- **Revenue:** `OD_REV_USD` is a priority availability question. If it is realized revenue from the forecast period and becomes known afterward, it leaks future information. A correctly available historical revenue feature would need a separate, justified lag and availability check. Correlation alone does not prove leakage or a deterministic relationship.
- **Calendar and routes:** Departure month/year and route identifiers are candidates only if the task supplies them at prediction time. Airport IDs are potentially meaningful categories, unlike a purely exported row index.
- **Timestamp:** A date/time format does not establish the event recorded by `SF_LOAD_TMS`. Until its role is confirmed, I propose withholding it from predictors while retaining it for possible availability auditing.
- **Airport metadata:** Verify the airport-key meaning, historical availability, and snapshot timing before adopting reference attributes. A current reference file is not automatically a historically correct snapshot.

**Draft takeaway:** Therefore, availability and provenance determine my leakage decisions. An unclear field stays unresolved until the team confirms it; a strongly associated field is not automatically a leak, and a weakly associated field is not automatically safe.

**Local conclusion — complete after confirmation:** Record the confirmed forecast point, which fields are available then, which are unavailable, and which still need a definition. Do not turn provisional concerns into established findings.


## 5. Relationships that bear on model choice

**What I am checking:** categorical cardinality, differences between route/category groups, possible numerical relationships, and genuine repeated observations over time. All target-aware diagnostics below use `eda_pax`, not held-out outcomes.

Numerical airport codes are not continuous measurements. Native categorical support depends on the estimator and configuration; tree models do not automatically accept arbitrary text identifiers. Any encoding must be fitted within the training process.


In [ ]:
category_summary = []
for column in ["ORG_AIRPRT", "DES_AIRPRT"]:
    category_summary.append({
        "feature": column,
        "distinct_categories": eda_pax[column].nunique(dropna=False),
    })
    grouped_target = (
        eda_pax.groupby(column, dropna=False)[TARGET_COLUMN]
        .agg(records="size", target_mean="mean", target_median="median")
    )
    print(column)
    # Aggregate groups only; no raw passenger records are displayed.
    display(grouped_target.nlargest(12, "records"))
display(pd.DataFrame(category_summary))

print("Distinct observed origin/destination pairs:")
print(eda_pax.groupby(["ORG_AIRPRT", "DES_AIRPRT"], dropna=False).ngroups)


In [ ]:
# Association with suspect revenue is diagnostic, not permission to use it.
if TARGET_COLUMN != "OD_REV_USD" and pd.api.types.is_numeric_dtype(eda_pax["OD_REV_USD"]):
    related_measures = eda_pax[[TARGET_COLUMN, "OD_REV_USD"]].dropna()
    display(related_measures.corr())
    if not related_measures.empty:
        related_measures.plot.hexbin(
            x="OD_REV_USD", y=TARGET_COLUMN, gridsize=30, mincnt=1,
            title="Exploration-only association; revenue availability unconfirmed",
        )
        plt.show()
else:
    print("Review a distinct, confirmed numerical relationship if applicable.")


In [ ]:
calendar_view = eda_pax.assign(
    departure_period_for_eda=eda_period,
    departure_month_for_eda=departure_month.loc[eda_pax.index],
)
repeated_route_periods = calendar_view.duplicated(
    subset=["ORG_AIRPRT", "DES_AIRPRT", "departure_period_for_eda"]
)
display(pd.Series({
    "distinct_departure_periods": eda_period.nunique(),
    "repeated_route_period_records": repeated_route_periods.sum(),
}))

period_summary = (
    calendar_view.groupby("departure_period_for_eda")[TARGET_COLUMN]
    .agg(records="size", target_mean="mean", target_median="median")
    .sort_index()
)
display(period_summary)
if not period_summary.empty:
    period_summary["target_mean"].plot(
        marker="o", title="Exploration-only mean across records by departure period"
    )
    plt.ylabel("Mean of the target measure")
    plt.show()

monthly_summary = (
    calendar_view.groupby("departure_month_for_eda")[TARGET_COLUMN]
    .agg(records="size", target_mean="mean", target_median="median")
    .sort_index()
)
display(monthly_summary)
# These summaries mix routes/record types. They do not establish seasonality,
# a causal time trend, or a valid single-route forecasting series.


In [ ]:
# Audit candidate join keys without joining or exposing individual records.
airport_keys = airports["AIRPRT_CD"].dropna()
display(pd.Series({
    "airport_records_with_missing_key": airports["AIRPRT_CD"].isna().sum(),
    "airport_records_with_repeated_key": airports["AIRPRT_CD"].duplicated(keep=False).sum(),
    "exploration_origin_match_share": eda_pax["ORG_AIRPRT"].isin(airport_keys).mean(),
    "exploration_destination_match_share": eda_pax["DES_AIRPRT"].isin(airport_keys).mean(),
}))
# Low match rates may reflect different code systems or types. Verify the key;
# do not coerce codes blindly, discard unmatched rows, or permit row multiplication.


**Draft takeaway:** Therefore, my model-family view should follow the observed relationships rather than the column names alone. Many categories require a deliberate encoding plan; rare groups may make category summaries unstable. Strong association with revenue does not establish that revenue is available as a predictor.

Departure-period observations may support temporal analysis, but repeated route-period records could be updates, snapshots, or distinct subgroups. Confirm their meaning before aggregation. The across-record average can change because the route mix changes; it does not prove seasonal demand for a fixed route. A load timestamp alone is not a forecasting time axis.

**Local conclusion — complete after inspecting:** Describe observed category differences, cardinality considerations, linear/nonlinear evidence if any, and whether enough comparable history exists for the intended forecasting task. State which model families those observations support or challenge, and agree validation with Kimberly/the lead.


## 6. My deep-dive — leakage hand-off

My responsibility is to review every suspect column and give a reason for each decision. The next cell creates a per-column audit using source names and authored proposals only. It is a review worksheet, not an automatically approved feature set. Unexpected fields remain unclassified until their definitions are checked.

My review also covers preprocessing, historical features, and joins:

- Fit learned imputation, scaling, encoding, and feature selection on training records only. Target-based encoding requires protection against using a record's own outcome and validation outcomes.
- Build historical/lagged features only from observations available before each forecast. A full-dataset route average can expose held-out or future outcomes.
- Keep records belonging to the same held-out departure period out of earlier training. Account for duplicate snapshots and the intended generalization goal with the validation owner.
- Verify airport-key uniqueness, matching, and historical availability. A reference join must not multiply passenger records or introduce later information.
- Separate an outcome label from its predictors. A target retained for training is not an approved input feature.


In [ ]:
passenger_proposals = {
    "DEP_MONTH": ("engineer (provisional)", "Calendar candidate if known at forecast time; representation depends on the agreed model."),
    "DEP_YEAR": ("keep (provisional)", "Departure-period candidate if known; use forecasting-realistic validation."),
    "ORG_AIRPRT": ("keep / engineer (provisional)", "Origin identifier; verify code meaning and use appropriate categorical treatment."),
    "DES_AIRPRT": ("keep / engineer (provisional)", "Destination identifier; verify code meaning and use appropriate categorical treatment."),
    "OD_PAX": ("needs clarification", "Proposed passenger target; confirm definition and availability if its role changes."),
    "OD_REV_USD": ("needs clarification", "Exclude as a leak if this is realized forecast-period revenue unavailable at the prediction point."),
    "SF_LOAD_TMS": ("drop (provisional)", "Event and availability unconfirmed; retain outside predictors for possible availability auditing."),
}
airport_proposals = {
    "AIRPRT_CD": ("id / engineer (provisional)", "Candidate airport join key/category; confirm code system, uniqueness, and availability."),
    "AIRPRT_NM": ("needs clarification", "Assess whether the name adds useful available information beyond the airport key."),
    "CITY_METRO_IATA_CD": ("keep / engineer (provisional)", "Candidate location category; confirm meaning, code system, and availability."),
    "STATE_PROVNC_CD": ("keep / engineer (provisional)", "Candidate location category with gaps; confirm missingness meaning and availability."),
    "CNTRY_CD": ("keep / engineer (provisional)", "Candidate location category; validate definitions, missingness, and availability."),
    "WRLD_AREA_DOT_CD": ("needs clarification", "Obtain the code definition before deciding its modeling role."),
    "SF_LOAD_TMS": ("drop (provisional)", "Clarify the reference snapshot event and date; preserve for availability auditing."),
    "LNGST_RUNWAY_FT_QTY": ("keep (provisional)", "Confirm measurement definition, units, and historical availability."),
    "ELEVATN_FT_QTY": ("keep (provisional)", "Confirm measurement definition, units, and historical availability."),
}
for column in [
    "LAT_DEGREE_QTY", "LAT_MINUTE_QTY", "LAT_SECOND_QTY", "LAT_HEMSPHR_CD",
    "LNGTD_DEGREE_QTY", "LNGTD_MINUTE_QTY", "LNGTD_SECOND_QTY", "LNGTD_HEMSPHR_CD",
]:
    airport_proposals[column] = (
        "engineer (provisional)",
        "Verify coordinate definitions, units/directions, and availability before any combination or use.",
    )

audit_rows = []
for source_name, frame, proposals in [
    ("Passenger", pax, passenger_proposals),
    ("Airport", airports, airport_proposals),
]:
    for column in frame.columns:
        decision, reason = proposals.get(
            column,
            ("needs clarification", "Confirm this field's meaning and availability with the lead."),
        )
        if source_name == "Passenger" and column == TARGET_COLUMN:
            decision = "drop from predictors; retain as target"
            reason = "Keep the confirmed outcome as the label and exclude it from its own inputs."
        elif str(column).strip().lower().startswith("unnamed:") or str(column).strip().lower() == "index":
            decision = "id / drop (provisional)"
            reason = "Exclude if confirmed to be a pure export index; do not infer a business role from ordering."
        audit_rows.append({
            "Column / feature": f"{source_name}: {column}",
            "Decision (keep / drop / engineer / leak / id)": decision,
            "Reason": reason,
        })

leakage_audit = pd.DataFrame(audit_rows)
display(leakage_audit)
# Edit proposals after team confirmation; this cell does not drop or engineer data.


### Provisional hand-off to Akshith

| Column / feature | Decision (keep / drop / engineer / leak / id) | Reason |
|---|---|---|
| `DEP_MONTH` | Engineer, provisionally | Candidate calendar information if available when forecasting; encoding needs the agreed model. |
| `DEP_YEAR` | Keep, provisionally | Candidate departure-period information; chronology must be respected for future-period evaluation. |
| `ORG_AIRPRT` | Keep / engineer, provisionally | Origin identifier with potential categorical value; confirm the code system. |
| `DES_AIRPRT` | Keep / engineer, provisionally | Destination identifier with potential categorical value; confirm the code system. |
| `OD_PAX` | Drop from predictors; retain as target if confirmed | The outcome must not be its own input. |
| `OD_REV_USD` | Needs clarification; leak / drop if unavailable | Determine whether it is realized after the forecast point or legitimately historical. |
| `SF_LOAD_TMS` | Drop from predictors, provisionally | Its event meaning and availability are unconfirmed; keep it available for provenance auditing. |
| Row-counter field | ID / drop if confirmed as an export index | A pure row index is not a business measurement. |
| Candidate airport attributes | Review each field before keeping/engineering | Confirm definitions, matching keys, missingness, and historical availability using the per-column audit. |

**Draft hand-off:** My main unresolved leakage question is whether the revenue measure would exist at the forecast point. The target must remain separate from input features. Route/calendar fields are candidates, while export indices and unclear processing timestamps should not enter predictors without justification. Airport attributes need a historically valid reference and a verified join. Learned preprocessing and historical aggregates must not use held-out or future outcomes.

**Local conclusion — complete before submission:** Replace provisional decisions with lead-confirmed decisions and reasons, and record remaining questions explicitly. Keep this hand-off qualitative. Send only the words-only hand-off to the lead; do not attach local outputs.


## 7. My model-family view

My provisional candidate is **gradient-boosted regression**, with a **regularized linear model such as Ridge** as a comparison baseline. Boosted trees are worth evaluating for nonlinear effects and interactions that route/calendar data may contain. A linear baseline provides a simpler comparison when additive effects are plausible. These are model capabilities, not empirical findings about this dataset.

Airport identifiers require appropriate categorical handling in either approach. Native categorical support depends on the implementation and configuration; arbitrary numeric codes must not silently become continuous measurements. I am not adding a library or training models in this notebook.

Before presenting a final choice, I need to connect it to my own local observations:

| Required justification | My qualitative evidence to complete |
|---|---|
| Observed relationship shape | Describe which plots support additive effects, nonlinear effects, or interactions, or state that evidence is inconclusive. |
| Feature representation and missingness | Explain how the proposed family would handle the confirmed categories, valid numerical inputs, and documented gaps. |
| Practical suitability | Explain interpretability, computation, maintenance, or another relevant constraint without claiming unmeasured performance. |
| Validation and limits | State the confirmed prediction task, the agreed validation logic, and any remaining data limitations. |

**Draft takeaway:** Therefore, my final recommendation must use available-at-forecast features and be supported by observed EDA evidence. I will not claim that nonlinear modeling is necessary, a linear model fails, or one model wins without the corresponding evidence. A family recommendation remains provisional while the target definition, field availability, or relationship evidence is unresolved.

**Local conclusion — complete after inspection:** Name one recommended family and give concrete qualitative reasons from this notebook. State the main limitation and the agreed validation approach. Do not include numerical results or unsupported model scores.


## References and local completion

These references explain methods; they do not establish findings about the AA files.

- [scikit-learn: leakage and preprocessing pitfalls](https://scikit-learn.org/stable/common_pitfalls.html)
- [scikit-learn: missing-value handling](https://scikit-learn.org/stable/modules/impute.html)
- [scikit-learn: time-series cross-validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [scikit-learn: linear models](https://scikit-learn.org/stable/modules/linear_model.html)
- [scikit-learn: tree ensembles](https://scikit-learn.org/stable/modules/ensemble.html)

Before submitting, complete the configuration with the team, run the notebook locally in order, write the **Local conclusion** answers, and resolve provisional hand-off decisions. Review Markdown for pasted values or embedded images. Then **Kernel → Restart & Clear Output**, save, and confirm git shows only this notebook.

The PR base is `team-02/main`. Akshith reviews my work; I review Aidan's PR. The lead handles merging. This authored, unexecuted notebook is preparation, not a completed EDA submission.
